# QPUF Test – 1-Qubit Haar Random Unitary, Two-Stage QPE

**Protocol (mid-circuit measurement, Qiskit)**

| | Stage 1 | Stage 2 |
|--|--|--|
| Precision qubits | `prec1` (n_prec) | `prec2` (n_prec) |
| Target qubit | shared (1 qubit) | same, after collapse |
| Measurement | mid-circuit → `c1` | final → `c2` |

The 1-qubit Haar-random unitary U is generated via ZYZ Euler decomposition.

**Workflow**
- **A** Local simulation (AerSimulator) — verify the protocol and circuit  
- **B** Cost estimation — transpile to IQM native gates, estimate AWS bill  
- **C** Retrieve hardware job (submitted via `submit_iqm_garnet.py`) and analyse results

Hardware submission is handled by `submit_iqm_garnet.py`, which logs the job ID
and metadata to `jobs_log.jsonl`.  Section C reads that file to retrieve the job.

In [ ]:
import json
import os

import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator

from braket.aws import AwsQuantumTask

## Helper Functions

In [ ]:
def haar_random_1qubit_matrix(rng=None):
    """
    Returns (U, angles) where U is a 2×2 Haar-random unitary matrix
    via ZYZ Euler decomposition: U = Rz(phi) @ Ry(theta) @ Rz(lam).

    Sampling:
      phi, lam ~ Uniform[0, 2π)
      theta = 2 * arccos(sqrt(u)),  u ~ Uniform[0, 1]   (Haar-correct)
    """
    if rng is None:
        rng = np.random.default_rng()

    phi   = rng.uniform(0, 2 * np.pi)
    lam   = rng.uniform(0, 2 * np.pi)
    u     = rng.uniform(0, 1)
    theta = 2 * np.arccos(np.sqrt(u))

    def Rz(a):
        return np.array([[np.exp(-1j * a / 2), 0.0],
                         [0.0,                 np.exp(1j * a / 2)]])

    def Ry(a):
        return np.array([[ np.cos(a / 2), -np.sin(a / 2)],
                         [ np.sin(a / 2),  np.cos(a / 2)]])

    U = Rz(phi) @ Ry(theta) @ Rz(lam)
    return U, dict(phi=phi, theta=theta, lam=lam)


def build_qpe_circuit(n_prec: int, angles: dict) -> QuantumCircuit:
    """
    Build a 1-qubit QPE sub-circuit using direct controlled-RzRyRz gates.

    U = Rz(phi) @ Ry(theta) @ Rz(lam)  (Haar-random, ZYZ decomposition)
    U^(2^k) is realised by repeating the gate sequence 2^k times.

    Qubit layout: [prec[0], ..., prec[n_prec-1], target]
    """
    n_targ = 1
    total  = n_prec + n_targ

    qc   = QuantumCircuit(total, name='QPE')
    prec = list(range(n_prec))
    targ = n_prec

    phi   = angles['phi']
    theta = angles['theta']
    lam   = angles['lam']

    qc.h(prec)

    for k in range(n_prec):
        ctrl = prec[k]
        for _ in range(2 ** k):
            qc.crz(lam,   ctrl, targ)
            qc.cry(theta, ctrl, targ)
            qc.crz(phi,   ctrl, targ)

    iqft = QFTGate(n_prec).inverse()
    qc.append(iqft, prec)

    return qc


def cyclic_distance(a: int, b: int, n_prec: int) -> int:
    """Cyclic distance |a - b| mod 2^n_prec."""
    M    = 2 ** n_prec
    diff = abs(a - b)
    return min(diff, M - diff)


def parse_outcome(bitstring: str, n_prec: int):
    """
    Parse a Qiskit counts key with two named ClassicalRegisters.

    Qiskit formats keys as 'c2_bits c1_bits' (space-separated).
    Returns (m1, m2) as integers.
    """
    parts = bitstring.split(' ')
    return int(parts[1], 2), int(parts[0], 2)


def analyse_counts(counts: dict, n_prec: int, delta: int, label: str = ''):
    """Print acceptance stats and top outcomes for a counts dict."""
    total    = sum(counts.values())
    accepted = sum(
        count for outcome, count in counts.items()
        if cyclic_distance(*parse_outcome(outcome, n_prec), n_prec) <= delta
    )
    acc_rate = accepted / total

    prefix = f'[{label}] ' if label else ''
    print(f'{prefix}Total shots        : {total}')
    print(f'{prefix}Accepted (dist ≤ {delta}) : {accepted}')
    print(f'{prefix}Acceptance rate    : {acc_rate:.4f}')

    print(f'\n{prefix}Top 10 outcomes:')
    print(f'  {{"bitstring":28s}}  m1  m2  dist  count')
    print(f'  {"-"*52}')
    for k, v in sorted(counts.items(), key=lambda x: -x[1])[:10]:
        m1, m2 = parse_outcome(k, n_prec)
        print(f'  {k!r:28s}  {m1:3d}  {m2:3d}  {cyclic_distance(m1,m2,n_prec):4d}  {v}')

    return total, accepted, acc_rate


def plot_m1_vs_m2(counts: dict, n_prec: int, n_shots: int, title_suffix: str = ''):
    """Scatter plot of m1 vs m2 phase estimates."""
    m1_vals, m2_vals, weights = [], [], []
    for outcome, count in counts.items():
        m1, m2 = parse_outcome(outcome, n_prec)
        m1_vals.append(m1)
        m2_vals.append(m2)
        weights.append(count)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(m1_vals, m2_vals, s=np.array(weights) * 2.0, alpha=0.6)
    ax.plot([0, 2**n_prec - 1], [0, 2**n_prec - 1], 'r--', label='m1 = m2')
    ax.set_xlabel('m1  (Stage 1 QPE)')
    ax.set_ylabel('m2  (Stage 2 QPE)')
    title = f'Phase estimates – 1-qubit Haar unitary\n(n_prec={n_prec}, n_shots={n_shots})'
    if title_suffix:
        title += f'  {title_suffix}'
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_acceptance_vs_delta(counts: dict, n_prec: int, n_shots: int, delta: int):
    """Plot acceptance rate as a function of the delta threshold."""
    total       = sum(counts.values())
    delta_range = list(range(0, 2**n_prec + 1))
    acc_rates   = []

    for d in delta_range:
        acc = sum(
            count for outcome, count in counts.items()
            if cyclic_distance(*parse_outcome(outcome, n_prec), n_prec) <= d
        )
        acc_rates.append(acc / total)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(delta_range, acc_rates, marker='o', markersize=4)
    ax.axvline(delta, color='r', linestyle='--', label=f'current delta={delta}')
    ax.set_xlabel('delta  (cyclic distance threshold)')
    ax.set_ylabel('acceptance rate')
    ax.set_title(f'Acceptance rate vs delta  (n_prec={n_prec}, n_shots={n_shots})')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print('\ndelta → acceptance rate')
    for d, r in zip(delta_range, acc_rates):
        print(f'  delta={d:3d}: {r:.4f}')

## Parameters

These must match the values used in `submit_iqm_garnet.py` (specifically `SEED` and `N_PREC`)
so that the local simulation and hardware run use the identical circuit.

In [ ]:
n_prec  = 2      # precision qubits per QPE stage (must match submit_1qubit_qpu.py N_PREC)
n_targ  = 1      # 1 target qubit → 2×2 unitary
n_shots = 1000   # shots for local simulation
delta   = 2      # cyclic-distance acceptance threshold
seed    = 42     # must match submit_1qubit_qpu.py SEED

rng = np.random.default_rng(seed=seed)

# Generate Haar-random 1-qubit unitary
unitary, angles = haar_random_1qubit_matrix(rng=rng)

print(f'n_prec={n_prec}, n_targ={n_targ}, delta={delta}')
print(f'\nEuler angles:  phi={angles["phi"]:.4f}  theta={angles["theta"]:.4f}  lam={angles["lam"]:.4f}')

## Build Two-Stage QPE Circuit

```
target  ──[ry/rz init]──────[QPE1 target]──────────────────────[QPE2 target]──
prec1   ──────────────────[QPE1 prec]──[M]→c1───────────────────────────────────
prec2   ────────────────────────────────────────────────[QPE2 prec]──[M]→c2──
```

`[M]→c1` is a **mid-circuit measurement**: collapses `prec1` and projects `target`
onto the corresponding eigenstate of U. Stage 2 therefore always sees the same
eigenstate that Stage 1 just measured.

In [ ]:
# ── Registers ─────────────────────────────────────────────────────────────
targ_reg  = QuantumRegister(n_targ, 'target')
prec1_reg = QuantumRegister(n_prec, 'prec1')
prec2_reg = QuantumRegister(n_prec, 'prec2')
c1        = ClassicalRegister(n_prec, 'c1')   # Stage-1 QPE outcome
c2        = ClassicalRegister(n_prec, 'c2')   # Stage-2 QPE outcome

qc = QuantumCircuit(targ_reg, prec1_reg, prec2_reg, c1, c2)

# ── Target: random single-qubit state ─────────────────────────────────────
init_rng   = np.random.default_rng(seed=99)
theta0, phi0 = init_rng.uniform(0, np.pi), init_rng.uniform(0, 2 * np.pi)
qc.ry(theta0, targ_reg[0])
qc.rz(phi0,   targ_reg[0])

# ── Stage 1: first QPE ────────────────────────────────────────────────────
qpe1 = build_qpe_circuit(n_prec, angles)
qc.append(qpe1, list(prec1_reg) + list(targ_reg))

# ── Mid-circuit measurement → collapses target onto an eigenstate of U ────
qc.measure(prec1_reg, c1)
qc.barrier(label='collapse')

# ── Stage 2: second QPE on the collapsed target ───────────────────────────
qpe2 = build_qpe_circuit(n_prec, angles)
qc.append(qpe2, list(prec2_reg) + list(targ_reg))

# ── Final measurement ─────────────────────────────────────────────────────
qc.measure(prec2_reg, c2)

total_qubits = qc.num_qubits
print(f'Total qubits   : {total_qubits}  (2×{n_prec} prec + {n_targ} target)')
print(f'Classical bits : {qc.num_clbits}')
print(f'IQM Garnet budget check: {"✓ OK (≤ 20)" if total_qubits <= 20 else "✗ Too many qubits"}')

ops = qc.decompose(reps=1).count_ops()
print(f'\nGate counts (1 decomp): {dict(ops)}')

qc.decompose(reps=1).draw(output='mpl', fold=80)

## Section A: Local Simulation

Run the circuit on `AerSimulator` — supports mid-circuit measurements natively.
Each shot independently:
1. Runs Stage-1 QPE, **measures** `prec1` → collapses `target` onto an eigenstate of U  
2. Runs Stage-2 QPE on the *same collapsed* `target`, measures `prec2`

Because the target is projected onto a phase eigenstate after Stage 1, both
measurements track the same eigenvalue → `m1 ≈ m2` on every shot.

In [ ]:
simulator  = AerSimulator()
transpiled = transpile(qc, simulator)
job        = simulator.run(transpiled, shots=n_shots)
result     = job.result()
counts     = result.get_counts()

print(f'Unique outcomes : {len(counts)}')

In [ ]:
total, accepted, acc_rate = analyse_counts(counts, n_prec, delta, label='Simulator')

print('\nTheoretical QPE bins (ideal):')
for ev in np.linalg.eigvals(unitary):
    phase = np.angle(ev) / (2 * np.pi)
    if phase < 0:
        phase += 1
    print(f'  phase={phase:.4f}  ideal bin={round(phase * 2**n_prec) % 2**n_prec}')

In [ ]:
plot_m1_vs_m2(counts, n_prec, n_shots, title_suffix='[Local Simulator]')

In [ ]:
plot_acceptance_vs_delta(counts, n_prec, n_shots, delta)

## Section B: Cost Estimation for IQM Garnet

Before submitting to hardware:
1. Decompose and transpile the circuit to IQM Garnet's native gate set (`r`, `rz`, `cz`)  
2. Inspect depth and 2-qubit gate count (dominant noise source)  
3. Estimate the Amazon Braket bill

IQM Garnet pricing (Amazon Braket): **$0.30/task + $0.00145/shot**

In [ ]:
# Transpile one QPE stage in isolation and scale by 2
# (avoids optimizer moving gates across the mid-circuit measurement boundary)
qpe_single = build_qpe_circuit(n_prec, angles)
qpe_single.measure_all()

iqm_basis      = ['r', 'rz', 'cz']
qpe_transpiled = transpile(qpe_single, basis_gates=iqm_basis, optimization_level=1)

ops   = qpe_transpiled.count_ops()
n_cz  = ops.get('cz', 0)
n_1q  = sum(v for k, v in ops.items() if k not in ('cz', 'measure', 'barrier'))
depth = qpe_transpiled.depth()

print('=== Single QPE stage — transpiled to IQM native gates (approx.) ===')
print(f'  CZ gates     : {n_cz:4d}   → both stages ≈ {n_cz*2}')
print(f'  1-qubit gates: {n_1q:4d}   → both stages ≈ {n_1q*2}')
print(f'  Depth        : {depth:4d}   → both stages ≈ {depth*2}')
print(f'  Qubits used (full circuit) : {qc.num_qubits}  (IQM Garnet has 20)')

TASK_FEE = 0.30
SHOT_FEE = 0.00145

print('\n=== Estimated cost on IQM Garnet ===')
print(f'  {"Shots":>6}   Task fee   Shot fee   Total')
print(f'  {"-"*45}')
for shots in [100, 250, 500, 1000, 2000]:
    total_cost = TASK_FEE + shots * SHOT_FEE
    print(f'  {shots:>6}    ${TASK_FEE:.2f}      ${shots * SHOT_FEE:.4f}     ${total_cost:.2f}')

print('\n⚠  Verify current pricing at https://aws.amazon.com/braket/pricing/')
print(f'\nRecommendation: start with 100 shots (${TASK_FEE + 100*SHOT_FEE:.2f}) to verify connectivity.')

## Section C: Retrieve Hardware Job and Analyse

Jobs are submitted via `submit_iqm_garnet.py`, which appends a JSON record to
`jobs_log.jsonl` in the same directory.  Each record contains the job ID,
submission timestamp, device, n_prec, n_shots, seed, and Euler angles.

**Steps**
1. Load and display available jobs from `jobs_log.jsonl`  
2. Select a job (default: latest)  
3. Retrieve the result and run the same analysis as Section A

In [ ]:
LOG_FILE = os.path.join(os.getcwd(), 'jobs_log.jsonl')

if not os.path.exists(LOG_FILE):
    raise FileNotFoundError(
        f'Job log not found: {LOG_FILE}\n'
        'Run submit_1qubit_qpu.py first to submit a job.'
    )

with open(LOG_FILE) as f:
    job_records = [json.loads(line) for line in f if line.strip()]

print(f'Found {len(job_records)} job record(s) in {LOG_FILE}\n')
print(f'  {"#":>3}  {"datetime":25s}  {"device":8s}  {"n_prec":7s}  {"n_shots":7s}  job_id')
print(f'  {"-"*85}')
for i, rec in enumerate(job_records):
    print(f'  {i:>3}  {rec["datetime"]:25s}  {rec["device"]:8s}  '
          f'{rec["n_prec"]:7d}  {rec["n_shots"]:7d}  {rec["job_id"]}')

In [ ]:
# Set JOB_INDEX to the index shown above, or -1 to use the latest job
JOB_INDEX = -1

selected = job_records[JOB_INDEX]

hw_job_id    = selected['job_id']
hw_device    = selected['device']
hw_n_prec    = selected['n_prec']
hw_n_shots   = selected['n_shots']
hw_angles    = selected['angles']
hw_submitted = selected['datetime']

task = AwsQuantumTask(arn = hw_job_id)


print(f'Selected job:')
print(f'  Job ID    : {hw_job_id}')
print(f'  Submitted : {hw_submitted}')
print(f'  Device    : {hw_device}')
print(f'  n_prec    : {hw_n_prec}')
print(f'  n_shots   : {hw_n_shots}')
print(f'  Angles    : phi={hw_angles["phi"]:.6f}  '
      f'theta={hw_angles["theta"]:.6f}  lam={hw_angles["lam"]:.6f}')

In [ ]:
from qiskit_braket_provider import BraketProvider

provider    = BraketProvider()
iqm_backend = provider.get_backend(hw_device)

print(f'Retrieving job {hw_job_id} …')
job_hw    = iqm_backend.retrieve_job(hw_job_id)
counts_hw = job_hw.result().get_counts()

print(f'Retrieved {sum(counts_hw.values())} shots with {len(counts_hw)} unique outcomes.')

In [ ]:
# Reconstruct the unitary from the logged angles for eigenvalue reference
def angles_to_unitary(angles):
    phi, theta, lam = angles['phi'], angles['theta'], angles['lam']
    def Rz(a): return np.array([[np.exp(-1j*a/2), 0], [0, np.exp(1j*a/2)]])
    def Ry(a): return np.array([[np.cos(a/2), -np.sin(a/2)], [np.sin(a/2), np.cos(a/2)]])
    return Rz(phi) @ Ry(theta) @ Rz(lam)

hw_unitary = angles_to_unitary(hw_angles)

total_hw, accepted_hw, acc_rate_hw = analyse_counts(
    counts_hw, hw_n_prec, delta, label='Hardware'
)

print('\nTheoretical QPE bins (ideal):')
for ev in np.linalg.eigvals(hw_unitary):
    phase = np.angle(ev) / (2 * np.pi)
    if phase < 0:
        phase += 1
    print(f'  phase={phase:.4f}  ideal bin={round(phase * 2**hw_n_prec) % 2**hw_n_prec}')

In [ ]:
plot_m1_vs_m2(counts_hw, hw_n_prec, hw_n_shots, title_suffix=f'[{hw_device} hardware]')

In [ ]:
plot_acceptance_vs_delta(counts_hw, hw_n_prec, hw_n_shots, delta)

In [ ]:
# ── Side-by-side comparison: simulator vs hardware ────────────────────────
if hw_n_prec == n_prec:
    print('=== Simulator vs Hardware comparison ===')
    print(f'  {"":30s}  {"Simulator":>12s}  {"Hardware":>12s}')
    print(f'  {"-"*58}')
    print(f'  {"Shots":30s}  {total:>12d}  {total_hw:>12d}')
    print(f'  {f"Accepted (dist ≤ {delta})": 30s}  {accepted:>12d}  {accepted_hw:>12d}')
    print(f'  {"Acceptance rate":30s}  {acc_rate:>12.4f}  {acc_rate_hw:>12.4f}')
else:
    print('n_prec mismatch between simulator and hardware runs — skipping comparison.')